# 13. Despliegue de Modelos de Machine Learning 🚀

Este notebook cubre el proceso completo de despliegue de modelos: serialización, validación pre-deploy, API REST con FastAPI, Docker y versionamiento.

✨ **Highlights:**
- Serialización y validación del modelo antes del despliegue
- API REST con FastAPI: endpoints `/predict` y `/health`
- Manejo de errores y validación de entrada con Pydantic
- Contenedorización con Docker y plantilla de `docker-compose`
- Versionamiento de modelos con metadata JSON

## Objetivo
- Aprender a serializar y cargar modelos entrenados.
- Crear una API REST de inferencia con FastAPI.
- Conocer las bases de contenedorización con Docker.
- Aplicar buenas prácticas de despliegue y versionamiento.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado al menos los notebooks [01](./01_intro_machine_learning.ipynb) y [03](./03_modelos_clasicos_ml.ipynb).

- Conceptos de entrenamiento y evaluación de modelos.

> ⚠️ **Dependencias adicionales:** `pip install fastapi uvicorn nest_asyncio`

## 1. Introducción teórica

El despliegue es el paso final del ciclo de vida de ML: llevar un modelo entrenado a producción para que pueda ser usado por aplicaciones reales.

**Etapas del despliegue:**
1. Serialización del modelo (guardar/cargar).
2. Creación de una API de inferencia.
3. Contenedorización (Docker).
4. Despliegue en la nube o servidor.

| Herramienta | Uso |
|-------------|-----|
| **joblib / pickle** | Serialización de modelos scikit-learn |
| **model.save()** | Serialización de modelos Keras |
| **FastAPI** | API REST de inferencia |
| **Docker** | Contenedorización para portabilidad |
| **MLflow / DVC** | Versionamiento de modelos y experimentos |

## 2. Entrenar y guardar un modelo

In [1]:
import random
import numpy as np

# === Reproducibilidad ===
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

import warnings
warnings.filterwarnings('ignore')

# Entrenar modelo de ejemplo
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=SEED)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy en test: {acc:.3f}')
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

# Guardar modelo
MODEL_PATH = 'modelo_iris.joblib'
joblib.dump(clf, MODEL_PATH)
print(f'✅ Modelo guardado en: {MODEL_PATH}')

Accuracy en test: 0.900

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

✅ Modelo guardado en: modelo_iris.joblib


## 2b. Validación del modelo antes del despliegue

> ⚠️ Antes de desplegar, siempre verificar que el modelo cargado produce las mismas predicciones.

In [2]:
import os

# Verificar que el archivo existe y tiene un tamaño razonable
assert os.path.exists(MODEL_PATH), f'ERROR: modelo no encontrado en {MODEL_PATH}'
size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f'Archivo: {MODEL_PATH} ({size_kb:.1f} KB)')

# Cargar y verificar predicciones
del clf
modelo = joblib.load(MODEL_PATH)

# Smoke test: predicciones en muestra conocida
sample = X_test[:5]
preds_originales = y_pred[:5]
preds_cargadas = modelo.predict(sample)
assert np.all(preds_originales == preds_cargadas), 'ERROR: predicciones inconsistentes tras carga'
print(f'✅ Validación exitosa — predicciones consistentes tras carga del modelo')

# Probar predicción individual
sample_single = X[0:1]
pred = modelo.predict(sample_single)
prob = modelo.predict_proba(sample_single)
print(f'\nEjemplo de predicción:')
print(f'  Clase: {data.target_names[pred[0]]} (id={pred[0]})')
print(f'  Probabilidades: { {name: f"{p:.3f}" for name, p in zip(data.target_names, prob[0])} }')

Archivo: modelo_iris.joblib (163.8 KB)
✅ Validación exitosa — predicciones consistentes tras carga del modelo

Ejemplo de predicción:
  Clase: setosa (id=0)
  Probabilidades: {np.str_('setosa'): '1.000', np.str_('versicolor'): '0.000', np.str_('virginica'): '0.000'}


## 4. Crear y lanzar una API REST desde el notebook

Usaremos FastAPI y `nest_asyncio` para lanzar la API dentro del notebook.

In [3]:
import importlib, subprocess, sys

for pkg, import_name in [('fastapi', 'fastapi'), ('uvicorn', 'uvicorn'), ('nest_asyncio', 'nest_asyncio')]:
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} instalado ✅')

print('✅ Dependencias disponibles: fastapi, uvicorn, nest_asyncio')

✅ Dependencias disponibles: fastapi, uvicorn, nest_asyncio


In [4]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, validator
from typing import List, Dict
import joblib
import numpy as np
import nest_asyncio
import uvicorn
from threading import Thread

modelo = joblib.load('modelo_iris.joblib')
CLASS_NAMES = ['setosa', 'versicolor', 'virginica']
N_FEATURES = 4

app = FastAPI(title='Iris Classifier API', version='1.0', description='API de inferencia para clasificación de flores Iris')

class InputData(BaseModel):
    features: List[float]

    @validator('features')
    def validate_features(cls, v):
        if len(v) != N_FEATURES:
            raise ValueError(f'Se esperan exactamente {N_FEATURES} features, se recibieron {len(v)}')
        if not all(isinstance(f, (int, float)) for f in v):
            raise ValueError('Todos los features deben ser numéricos')
        if not all(0 <= f <= 20 for f in v):
            raise ValueError('Los valores de las features deben estar en rango [0, 20]')
        return v

class PredictionResponse(BaseModel):
    prediction: int
    class_name: str
    probabilities: Dict[str, float]

@app.post('/predict', response_model=PredictionResponse, summary='Predice la especie de Iris')
def predict(data: InputData):
    try:
        X = np.array(data.features).reshape(1, -1)
        pred = int(modelo.predict(X)[0])
        proba = modelo.predict_proba(X)[0]
        return PredictionResponse(
            prediction=pred,
            class_name=CLASS_NAMES[pred],
            probabilities={name: round(float(p), 4) for name, p in zip(CLASS_NAMES, proba)}
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Error de inferencia: {str(e)}')

@app.get('/health', summary='Health check')
def health():
    return {'status': 'ok', 'model': 'RandomForestClassifier', 'version': '1.0'}

# Lanzar API en Jupyter
nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

api_thread = Thread(target=run_api, daemon=True)
api_thread.start()
print('✅ API lanzada en http://localhost:8000')
print('📖 Documentación interactiva: http://localhost:8000/docs')

✅ API lanzada en http://localhost:8000
📖 Documentación interactiva: http://localhost:8000/docs


## 5. Probar la API (local en notebook y/o en contenedor Docker)

In [5]:
import requests
import time

BASE_URL = 'http://127.0.0.1:8000'

time.sleep(2)

def test_api(base_url, label=''):
    """Ejecuta una batería de pruebas contra la API en base_url."""
    tag = f'[{label}] ' if label else ''

    # 1. Health check
    try:
        resp = requests.get(f'{base_url}/health', timeout=3)
        resp.raise_for_status()
        print(f'{tag}✅ Health check: {resp.json()}')
    except requests.exceptions.ConnectionError:
        print(f'{tag}⚠️  API no disponible en {base_url}')
        return

    # 2. Predicción válida
    payload_ok = {'features': [5.1, 3.5, 1.4, 0.2]}
    resp = requests.post(f'{base_url}/predict', json=payload_ok, timeout=3)
    print(f'{tag}✅ Predicción ({resp.status_code}): {resp.json()}')

    # 3. Validación de entrada — debe devolver 422
    payload_bad = {'features': [5.1, 3.5]}
    resp_bad = requests.post(f'{base_url}/predict', json=payload_bad, timeout=3)
    msg = resp_bad.json().get('detail', [{}])
    msg = msg[0].get('msg', msg) if isinstance(msg, list) else msg
    print(f'{tag}⚠️  Validación incorrecta ({resp_bad.status_code}): {msg}')

    # 4. Benchmark de latencia
    latencies = []
    for _ in range(10):
        t0 = time.time()
        requests.post(f'{base_url}/predict', json=payload_ok, timeout=3)
        latencies.append((time.time() - t0) * 1000)
    print(f'{tag}⏱  Latencia — media: {np.mean(latencies):.1f} ms  '
          f'mín: {np.min(latencies):.1f} ms  máx: {np.max(latencies):.1f} ms')

# ── API en el notebook ─────────────────────────────────────────
print('=== API en el notebook ===')
test_api(BASE_URL, label='notebook')

# ── Misma batería contra el contenedor Docker (si está corriendo) ─
print('\n=== API en contenedor Docker ===')
test_api('http://localhost:8000', label='docker')

=== API en el notebook ===
[notebook] ✅ Health check: {'status': 'ok', 'model': 'RandomForestClassifier', 'version': '1.0'}
[notebook] ✅ Predicción (200): {'prediction': 0, 'class_name': 'setosa', 'probabilities': {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}}
[notebook] ⚠️  Validación incorrecta (422): Value error, Se esperan exactamente 4 features, se recibieron 2
[notebook] ⏱  Latencia — media: 12.0 ms  mín: 10.8 ms  máx: 14.5 ms

=== API en contenedor Docker ===
[docker] ✅ Health check: {'status': 'ok', 'model': 'RandomForestClassifier', 'version': '1.0'}
[docker] ✅ Predicción (200): {'prediction': 0, 'class_name': 'setosa', 'probabilities': {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}}
[docker] ⚠️  Validación incorrecta (422): Value error, Se esperan exactamente 4 features, se recibieron 2
[docker] ⏱  Latencia — media: 18.6 ms  mín: 12.0 ms  máx: 30.5 ms


## 6. Contenedorización con Docker

El objetivo es empaquetar el modelo entrenado, la API y sus dependencias en una imagen reproducible. Las siguientes celdas **escriben todos los archivos necesarios al disco** — no son plantillas, son los archivos reales listos para `docker build`.

Estructura del contexto de build:
```
deploy/
├── app.py                ← API FastAPI (producción, sin notebook)
├── modelo_iris.joblib    ← modelo exportado desde la sección 2
├── requirements.txt      ← dependencias con versiones exactas
├── Dockerfile            ← imagen python:3.10-slim con el modelo incluido
├── docker-compose.yml    ← orquestación con healthcheck
└── .dockerignore         ← excluye notebooks y cache del contexto
```

### 6a. Crear directorio de despliegue y copiar el modelo

In [6]:
import os
import shutil

DEPLOY_DIR = 'deploy'
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copiar el modelo ya validado al contexto de build
src = MODEL_PATH                           # 'modelo_iris.joblib'
dst = os.path.join(DEPLOY_DIR, src)
shutil.copy2(src, dst)

size_kb = os.path.getsize(dst) / 1024
print(f'✅ Modelo copiado a {dst}  ({size_kb:.1f} KB)')
print(f'   SHA256: ', end='')

import hashlib
with open(dst, 'rb') as f:
    print(hashlib.sha256(f.read()).hexdigest()[:16], '...')

✅ Modelo copiado a deploy/modelo_iris.joblib  (163.8 KB)
   SHA256: 61aa0a8e16794e18 ...


### 6b. Generar `app.py` — API de producción

Versión standalone del endpoint FastAPI: sin `nest_asyncio` ni hilos — lista para `uvicorn app:app`.

In [7]:
APP_CODE = '''\
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, validator
from typing import List, Dict
import joblib
import numpy as np

# ── Configuración ──────────────────────────────────────────────
MODEL_PATH  = "modelo_iris.joblib"
CLASS_NAMES = ["setosa", "versicolor", "virginica"]
N_FEATURES  = 4

# Cargar modelo al iniciar (una sola vez)
modelo = joblib.load(MODEL_PATH)

app = FastAPI(
    title="Iris Classifier API",
    version="1.0",
    description="Clasifica flores Iris en setosa, versicolor o virginica.",
)

# ── Esquemas de entrada/salida ─────────────────────────────────
class InputData(BaseModel):
    features: List[float]

    @validator("features")
    def validate_features(cls, v):
        if len(v) != N_FEATURES:
            raise ValueError(
                f"Se esperan exactamente {N_FEATURES} features, "
                f"se recibieron {len(v)}"
            )
        if not all(0.0 <= f <= 20.0 for f in v):
            raise ValueError("Cada feature debe estar en el rango [0, 20]")
        return v

class PredictionResponse(BaseModel):
    prediction: int
    class_name: str
    probabilities: Dict[str, float]

# ── Endpoints ──────────────────────────────────────────────────
@app.get("/health", summary="Health check")
def health():
    return {"status": "ok", "model": "RandomForestClassifier", "version": "1.0"}

@app.post("/predict", response_model=PredictionResponse, summary="Clasificar flor Iris")
def predict(data: InputData):
    try:
        X    = np.array(data.features).reshape(1, -1)
        pred = int(modelo.predict(X)[0])
        prob = modelo.predict_proba(X)[0]
        return PredictionResponse(
            prediction=pred,
            class_name=CLASS_NAMES[pred],
            probabilities={
                name: round(float(p), 4)
                for name, p in zip(CLASS_NAMES, prob)
            },
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Error de inferencia: {exc}")
'''

app_path = os.path.join(DEPLOY_DIR, 'app.py')
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

print(f'✅ {app_path} generado ({len(APP_CODE.splitlines())} líneas)')
print()
print(APP_CODE)

✅ deploy/app.py generado (61 líneas)

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, validator
from typing import List, Dict
import joblib
import numpy as np

# ── Configuración ──────────────────────────────────────────────
MODEL_PATH  = "modelo_iris.joblib"
CLASS_NAMES = ["setosa", "versicolor", "virginica"]
N_FEATURES  = 4

# Cargar modelo al iniciar (una sola vez)
modelo = joblib.load(MODEL_PATH)

app = FastAPI(
    title="Iris Classifier API",
    version="1.0",
    description="Clasifica flores Iris en setosa, versicolor o virginica.",
)

# ── Esquemas de entrada/salida ─────────────────────────────────
class InputData(BaseModel):
    features: List[float]

    @validator("features")
    def validate_features(cls, v):
        if len(v) != N_FEATURES:
            raise ValueError(
                f"Se esperan exactamente {N_FEATURES} features, "
                f"se recibieron {len(v)}"
            )
        if not all(0.0 <= f <= 20.0 for f in v):
    

### 6c. Generar `requirements.txt` con versiones exactas del entorno actual

In [8]:
import importlib.metadata

# Paquetes que necesita la API en producción
REQUIRED = ['fastapi', 'uvicorn', 'scikit-learn', 'numpy', 'joblib', 'pydantic']

lines = []
for pkg in REQUIRED:
    try:
        version = importlib.metadata.version(pkg)
        lines.append(f'{pkg}=={version}')
        print(f'  {pkg:20s} {version}')
    except importlib.metadata.PackageNotFoundError:
        lines.append(pkg)
        print(f'  {pkg:20s} (versión no encontrada — se instalará la última)')

req_content = '\n'.join(lines) + '\n'
req_path = os.path.join(DEPLOY_DIR, 'requirements.txt')
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(req_content)

print(f'\n✅ {req_path} generado')
print()
print(req_content)

  fastapi              0.136.1
  uvicorn              0.47.0
  scikit-learn         1.8.0
  numpy                2.4.5
  joblib               1.5.3
  pydantic             2.13.4

✅ deploy/requirements.txt generado

fastapi==0.136.1
uvicorn==0.47.0
scikit-learn==1.8.0
numpy==2.4.5
joblib==1.5.3
pydantic==2.13.4



### 6d. Generar `Dockerfile`

El modelo se copia **dentro de la imagen** en el paso `COPY modelo_iris.joblib .` — no se monta en runtime. Esto garantiza que imagen + modelo son una unidad atómica y reproducible.

In [9]:
DOCKERFILE = f'''\
# ── Base ───────────────────────────────────────────────────────
FROM python:3.10-slim

WORKDIR /app

# ── Dependencias (capa cacheada; se reconstruye solo si cambia) ─
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Modelo entrenado (copiado dentro de la imagen) ─────────────
# Versión fijada durante el build — no depende de volúmenes externos
COPY modelo_iris.joblib .

# ── Aplicación ─────────────────────────────────────────────────
COPY app.py .

# ── Puerto y healthcheck ────────────────────────────────────────
EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen(\'http://localhost:8000/health\')"

# ── Arranque ────────────────────────────────────────────────────
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

df_path = os.path.join(DEPLOY_DIR, 'Dockerfile')
with open(df_path, 'w', encoding='utf-8') as f:
    f.write(DOCKERFILE)

print(f'✅ {df_path} generado')
print()
print(DOCKERFILE)

✅ deploy/Dockerfile generado

# ── Base ───────────────────────────────────────────────────────
FROM python:3.10-slim

WORKDIR /app

# ── Dependencias (capa cacheada; se reconstruye solo si cambia) ─
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Modelo entrenado (copiado dentro de la imagen) ─────────────
# Versión fijada durante el build — no depende de volúmenes externos
COPY modelo_iris.joblib .

# ── Aplicación ─────────────────────────────────────────────────
COPY app.py .

# ── Puerto y healthcheck ────────────────────────────────────────
EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# ── Arranque ────────────────────────────────────────────────────
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]



### 6e. Generar `docker-compose.yml` y `.dockerignore`

In [10]:
COMPOSE = '''\
services:
  iris-api:
    build:
      context: .
      dockerfile: Dockerfile
    image: iris-classifier:latest
    container_name: iris-api
    ports:
      - "8000:8000"
    restart: unless-stopped
    healthcheck:
      test:
        - "CMD"
        - "python"
        - "-c"
        - "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
      interval: 30s
      timeout: 5s
      retries: 3
      start_period: 15s
    environment:
      - PYTHONUNBUFFERED=1
'''

DOCKERIGNORE = '''\
# Excluir del contexto de build para reducir tamaño y tiempo de copia
*.ipynb
.ipynb_checkpoints/
__pycache__/
*.pyc
*.pyo
.git/
.env
*.log

# Modelos versionados en el directorio padre (solo se usa el del COPY)
modelo_iris_v*.joblib
modelo_iris_v*_metadata.json
'''

compose_path = os.path.join(DEPLOY_DIR, 'docker-compose.yml')
ignore_path  = os.path.join(DEPLOY_DIR, '.dockerignore')

with open(compose_path, 'w', encoding='utf-8') as f:
    f.write(COMPOSE)
with open(ignore_path, 'w', encoding='utf-8') as f:
    f.write(DOCKERIGNORE)

print(f'✅ {compose_path} generado')
print(f'✅ {ignore_path} generado')
print()

# Mostrar árbol del contexto de build
import pathlib
print('📁 Contexto de build:')
for p in sorted(pathlib.Path(DEPLOY_DIR).iterdir()):
    size = p.stat().st_size
    unit = 'KB' if size > 1024 else 'B'
    size_fmt = f'{size/1024:.1f} {unit}' if size > 1024 else f'{size} {unit}'
    print(f'   {p.name:<30}  {size_fmt}')

✅ deploy/docker-compose.yml generado
✅ deploy/.dockerignore generado

📁 Contexto de build:
   .dockerignore                   264 B
   Dockerfile                      1.3 KB
   app.py                          2.2 KB
   docker-compose.yml              509 B
   modelo_iris.joblib              163.8 KB
   requirements.txt                97 B


### 6f. Construir y ejecutar la imagen Docker

La celda detecta si Docker está disponible y ejecuta el build automáticamente. Si Docker no está instalado, imprime los comandos equivalentes para ejecutar en la terminal.

In [11]:
import subprocess
import shutil as _shutil

IMAGE_NAME = 'iris-classifier:latest'
CONTAINER  = 'iris-api'

def docker_available():
    return _shutil.which('docker') is not None

def run(cmd, **kwargs):
    """Ejecuta un comando y muestra stdout/stderr en tiempo real."""
    print(f'$ {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

if docker_available():
    # ── 1. Verificar que el daemon está corriendo ───────────────
    ping = run(['docker', 'info', '--format', '{{.ServerVersion}}'])
    if ping.returncode != 0:
        print('⚠️  Docker instalado pero el daemon no responde — ábrelo y vuelve a ejecutar.')
    else:
        print(f'Docker daemon activo (v{ping.stdout.strip()}) ✅\n')

        # ── 2. Build ────────────────────────────────────────────
        print('🔨 Construyendo imagen...')
        build = run(
            ['docker', 'build', '-t', IMAGE_NAME, '.'],
            cwd=DEPLOY_DIR
        )

        if build.returncode == 0:
            print(f'\n✅ Imagen {IMAGE_NAME!r} construida')

            # Mostrar capas e inspección del tamaño
            run(['docker', 'image', 'inspect', IMAGE_NAME,
                 '--format', 'Tamaño: {{.Size}} bytes  |  OS: {{.Os}}/{{.Architecture}}'])

            # ── 3. Detener contenedor anterior si existe ────────
            run(['docker', 'rm', '-f', CONTAINER])

            # ── 4. Ejecutar contenedor ──────────────────────────
            print('\n🚀 Iniciando contenedor...')
            run([
                'docker', 'run', '-d',
                '--name', CONTAINER,
                '-p', '8000:8000',
                IMAGE_NAME
            ])

            # ── 5. Esperar y verificar health ───────────────────
            import time as _time
            _time.sleep(4)
            import urllib.request as _req
            try:
                with _req.urlopen('http://localhost:8000/health', timeout=5) as r:
                    print(f'\n✅ Health check del contenedor: {r.read().decode()}')
            except Exception as e:
                print(f'\n⚠️  Health check falló: {e}')
                run(['docker', 'logs', CONTAINER])
        else:
            print('❌ El build falló — revisa los errores de arriba.')
else:
    # ── Sin Docker: mostrar los comandos equivalentes ───────────
    print('ℹ️  Docker no encontrado en PATH.')
    print('   Ejecuta estos comandos en la terminal desde el directorio del proyecto:\n')
    print(f'   cd {DEPLOY_DIR}')
    print(f'   docker build -t {IMAGE_NAME} .')
    print(f'   docker run -d --name {CONTAINER} -p 8000:8000 {IMAGE_NAME}')
    print()
    print('   # Con docker-compose:')
    print(f'   cd {DEPLOY_DIR}')
    print('   docker compose up -d')
    print()
    print('   # Verificar que está corriendo:')
    print(f'   docker ps | grep {CONTAINER}')
    print(f'   curl http://localhost:8000/health')

ℹ️  Docker no encontrado en PATH.
   Ejecuta estos comandos en la terminal desde el directorio del proyecto:

   cd deploy
   docker build -t iris-classifier:latest .
   docker run -d --name iris-api -p 8000:8000 iris-classifier:latest

   # Con docker-compose:
   cd deploy
   docker compose up -d

   # Verificar que está corriendo:
   docker ps | grep iris-api
   curl http://localhost:8000/health


## 7. Versionamiento de modelos

En producción, es esencial versionar los modelos:

| Herramienta | Ventaja | Cuándo usar |
|-------------|---------|-------------|
| **MLflow** | Tracking de experimentos, registro de modelos | Equipos medianos/grandes |
| **DVC** | Control de versiones de datos y modelos | Integración con Git |
| **Convención manual** | Nombres con timestamp y métricas | Proyectos pequeños |

**Ejemplo de versionamiento manual:**

In [12]:
from datetime import datetime
import json

# Guardar modelo con versión
version = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = f'modelo_iris_v{version}.joblib'
joblib.dump(modelo, model_path)

# Guardar metadata
metadata = {
    'version': version,
    'model_type': 'RandomForestClassifier',
    'accuracy': accuracy_score(y_test, modelo.predict(X_test)),
    'features': list(data.feature_names),
    'target_names': list(data.target_names)
}

with open(f'modelo_iris_v{version}_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {model_path}')
print(f'Metadata: {json.dumps(metadata, indent=2)}')

Modelo guardado: modelo_iris_v20260515_210921.joblib
Metadata: {
  "version": "20260515_210921",
  "model_type": "RandomForestClassifier",
  "accuracy": 0.9,
  "features": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "target_names": [
    "setosa",
    "versicolor",
    "virginica"
  ]
}


## 8. Recomendaciones prácticas de despliegue

| Aspecto | Recomendación |
|---------|---------------|
| **Validación** | Verificar que el modelo cargado produce las mismas predicciones antes de desplegar |
| **Health check** | Siempre incluir `/health` — es el endpoint que monitorean los orquestadores (k8s) |
| **Validación de entrada** | Usar Pydantic validators para rechazar inputs malformados con error 422 |
| **Manejo de errores** | Envolver la inferencia en `try/except` y devolver HTTP 500 con detalle |
| **Versionamiento** | Guardar metadata (accuracy, features, fecha) junto al modelo |
| **Logging** | Registrar cada predicción con timestamp, input hash y resultado en producción |
| **Seguridad** | Agregar autenticación (API key o JWT) antes de exponer públicamente |
| **Docker** | Usar imagen `python:3.10-slim` — 3× más ligera que la imagen estándar |

> 💡 **Truco:** Para modelos de scikit-learn, `joblib` es más eficiente que `pickle` porque usa compresión y múltiples archivos para arrays NumPy grandes.

> 💡 **Truco FastAPI:** Los modelos `BaseModel` de Pydantic generan automáticamente la documentación OpenAPI — la URL `/docs` es completamente funcional sin configuración adicional.

## 9. Discusión y Conclusiones

**¿Qué aprendimos?**

- El despliegue es el paso que **convierte el modelo en valor real**: sin él, el modelo solo existe en notebooks.
- La **validación pre-deploy** (smoke test con assertions) previene regresiones silenciosas al recargar modelos.
- **FastAPI + Pydantic** permiten construir una API robusta en minutos, con validación automática, documentación interactiva y manejo de errores declarativo.
- El **health check** y el **benchmark de latencia** son las dos métricas más importantes para monitorear una API en producción.
- **Docker** garantiza que el entorno del notebook == entorno de producción, eliminando el clásico "funciona en mi máquina".
- El **versionamiento con metadata** permite auditar qué modelo tomó cada decisión y reproducir resultados históricos.

**Próximos pasos para llevar a producción:**
1. Agregar autenticación (Bearer token o API key)
2. Configurar logging estructurado (JSON) para enviar a un sistema de observabilidad
3. Empaquetar en Docker y desplegar en un servicio cloud (AWS ECS, GCP Cloud Run, etc.)
4. Configurar alertas si la latencia supera un umbral o el health check falla

¡Felicidades por completar el curso completo de Deep Learning! 🎉

## 10. Ejercicios Propuestos

1. **Ejercicio 1:** Agrega un endpoint `POST /predict/batch` que acepte una lista de observaciones (`List[List[float]]`) y devuelva una predicción por cada una.

2. **Ejercicio 2:** Entrena el modelo con `n_estimators=200` y vuelve a ejecutar el flujo completo. Verifica que el `requirements.txt` y el `Dockerfile` generados siguen siendo correctos sin modificarlos.

3. **Ejercicio 3:** Modifica el `Dockerfile` generado para usar un **build multi-etapa**: la primera etapa instala las dependencias y la segunda copia solo los artefactos necesarios para reducir el tamaño final de la imagen.

4. **Ejercicio 4 (Avanzado):** Configura MLflow para trackear el experimento de entrenamiento y guarda el modelo con `mlflow.sklearn.log_model()`. Adapta el `app.py` para cargarlo desde el MLflow Model Registry en lugar de un archivo `.joblib`.

## 10. Referencias y Recursos

- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [scikit-learn: Model Persistence](https://scikit-learn.org/stable/model_persistence.html)
- [Docker Getting Started](https://docs.docker.com/get-started/)
- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)

---

📎 **Notebook anterior:** [12. CPU, GPU y Metal](./12_cpu_gpu_metal.ipynb)  
📎 **Este es el último notebook del curso.** ¡Felicidades por completar el recorrido! 🎉